# EDA Module Demo

Demonstrates all functions in `caketool.eda` using the UCI Adult Income dataset (synthetic variant built from `sklearn` + `numpy`).

**Sections**
1. Setup & Data
2. Dataset Overview
3. Univariate Analysis
4. Bivariate Analysis
5. Multivariate Analysis
6. Data Quality

## 1. Setup & Data

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

from caketool import eda
from caketool.eda import EDAConfig

In [ ]:
# Load Adult Income dataset (binary classification)
adult = fetch_openml(name="adult", version=2, as_frame=True, parser="auto")
df = adult.frame.copy()

# Light cleaning
df["income"] = (df["class"] == ">50K").astype(int)
df = df.drop(columns=["class", "fnlwgt", "education-num"])
df.columns = df.columns.str.replace("-", "_")

# Introduce some missing values for quality demos
rng = np.random.default_rng(42)
for col in ["occupation", "native_country", "workclass"]:
    mask = rng.random(len(df)) < 0.05
    df.loc[mask, col] = np.nan

# Split train/test for PSI demo
split = int(len(df) * 0.7)
df_train = df.iloc[:split].reset_index(drop=True)
df_test = df.iloc[split:].reset_index(drop=True)

print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
# Time-series data for time series demo
t = np.arange(100)
df_ts = pd.DataFrame({
    'day': t,
    'price': 100 + 2*t + 15*np.sin(2*np.pi*t/30) + rng.normal(0, 3, 100)
})
df_ts.head(3)

## 2. Dataset Overview

In [ ]:
# Comprehensive per-column profile
eda.profile(df)

In [ ]:
# Hierarchically-clustered numeric correlation heatmap
eda.correlation_heatmap(df, method="pearson", mask_threshold=0.05)

In [ ]:
# Cramér's V heatmap for all categorical columns
cat_cols = [c for c in df.columns if df[c].dtype == "category" or df[c].nunique() < 20]
eda.cramers_v_heatmap(df, cat_cols=cat_cols, mask_threshold=0.05)

In [ ]:
# Top 5 extreme ages
eda.top_extreme_values(df, col="age", k=5, highest=True)

In [ ]:
# Pivot: income rate by education × workclass
eda.pivot_rate(df.dropna(subset=["workclass"]), index="education", columns="workclass", target="income")

## 3. Univariate Analysis

In [ ]:
# Histogram — basic
eda.plot_numeric_distribution(df["age"], nbins=40)

In [ ]:
# Histogram with outlier trim
eda.plot_numeric_distribution(df["capital_gain"], nbins=50, low_trim=0.0, high_trim=0.99)

In [ ]:
# KDE distribution
eda.plot_numeric_distribution(df["hours_per_week"], kde=True, high_trim=0.99)

In [ ]:
# Overlay: age distribution by income group
eda.plot_numeric_distribution(
    {"income=0": df[df["income"] == 0]["age"], "income=1": df[df["income"] == 1]["age"]},
    nbins=40,
)

In [ ]:
# Overlay KDE distribution
eda.plot_numeric_distribution(
    {"income=0": df[df["income"] == 0]["age"], "income=1": df[df["income"] == 1]["age"]},
    kde=True,
)

In [ ]:
# Percentile summary for hours_per_week
eda.summarize_numeric_series(df["hours_per_week"], step=10)

In [ ]:
# Bar count — categorical
eda.plot_categorical_frequency(df["education"], top_k=15, mode="barh")

In [ ]:
# Pie chart with Others grouping
eda.plot_categorical_frequency(df["native_country"], top_k=8, mode="pie")

In [ ]:
# Value counts table
eda.summarize_categorical_series(df["education"], top_k=15)

## 4. Bivariate Analysis

In [ ]:
rank_df = eda.rank_associations(df, target="income")
rank_df


In [ ]:
rank_df = eda.rank_associations(df, target="age")
rank_df


In [ ]:
eda.plot_scatter(df, x="age", y="hours_per_week", color_by="income")

In [ ]:
eda.plot_distribution_by_group(df, cat_col="income", num_col="age", mode="box", top_k=5)

In [ ]:
eda.plot_distribution_by_group(df, cat_col="income", num_col="hours_per_week", mode="violin", top_k=5)

In [ ]:
eda.plot_distribution_by_group(df, cat_col="income", num_col="age", mode="hist", top_k=3)

In [ ]:
eda.plot_category_heatmap(df, col1="income", col2="education", normalize=True)


In [ ]:
fig = eda.plot_time_series(df_ts, x="day", y="price", ma=7, band="std")
fig.show()

## 5. Multivariate Analysis

In [ ]:
# Scatter matrix of numeric features
num_feats = ["age", "hours_per_week", "capital_gain", "capital_loss"]
eda.scatter_matrix(df, columns=num_feats, color_by="income")

In [ ]:
# Parallel coordinates
eda.parallel_coordinates(df, dims=num_feats, color_by="age")

In [ ]:
# 3D scatter
eda.scatter_3d(df, x="age", y="hours_per_week", z="capital_gain", color_by="income")

In [ ]:
# Stacked bar: education composition by income (normalised)
eda.stacked_bar(df, x="income", category="education", normalize=True, top_k=8)

## 6. Data Quality

In [ ]:
# Missing value summary
eda.missing_summary(df)

In [ ]:
# Missing correlation heatmap
eda.missing_heatmap(df)

In [ ]:
# Duplicate rows check
dups = eda.duplicate_rows(df)
print(f"Duplicate rows: {len(dups)}")
dups.head()

In [ ]:
# Duplicate columns check
eda.duplicate_columns(df, threshold=90.0)

In [ ]:
# PSI report: train vs test
eda.psi_report(
    df_train,
    df_test,
    cols=["age", "hours_per_week", "capital_gain", "education", "occupation"],
)

In [ ]:
# PSI for a single column
val = eda.psi(df_train["age"], df_test["age"], buckets=10, method="quantile")
print(f"PSI (age, train vs test): {val:.4f}")

## 7. Custom Config

In [ ]:
# Override default styling via EDAConfig
cfg = EDAConfig(
    width=700,
    height=350,
    template="plotly_dark",
    top_k_categories=10,
)

eda.plot_numeric_distribution(df["age"], nbins=30, cfg=cfg)